# IBM Applied Data Science Capstone
## Falcon 9 landing prediction - Wikipedia web scraping

**Learner:** Djessi Jorge  
**Completed:** 4 August 2026

This notebook extracts historical Falcon 9 and Falcon Heavy launch-table content from a
fixed Wikipedia revision. The fixed revision prevents later page edits from changing the
results. A local completed snapshot is used during routine execution.

In [1]:
from io import StringIO
from pathlib import Path
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

WIKI_SNAPSHOT = (
    "https://en.wikipedia.org/w/index.php?title="
    "List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
)
LOCAL_SNAPSHOT = Path("spacex_web_scraped.csv")
RUN_LIVE_SCRAPE = False

In [2]:
def clean_text(value):
    value = re.sub(r"\[[^]]+\]", "", str(value))
    return " ".join(value.replace("\n", " ").split())

def scrape_launch_tables(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    frames = []
    for table in soup.select("table.wikitable"):
        headers = [clean_text(cell.get_text(" ")) for cell in table.select("tr th")]
        if not any("Flight" in header for header in headers):
            continue
        parsed = pd.read_html(StringIO(str(table)))[0]
        if isinstance(parsed.columns, pd.MultiIndex):
            parsed.columns = [clean_text(col[-1]) for col in parsed.columns]
        else:
            parsed.columns = [clean_text(col) for col in parsed.columns]
        frames.append(parsed)

    if not frames:
        raise ValueError("No launch tables were found in the supplied page.")
    return pd.concat(frames, ignore_index=True, sort=False)

In [3]:
if RUN_LIVE_SCRAPE:
    raw_scrape = scrape_launch_tables(WIKI_SNAPSHOT)
    raw_scrape.to_csv("wikipedia_launch_tables_raw.csv", index=False)
else:
    launch_history = pd.read_csv(LOCAL_SNAPSHOT)

print(f"Historical launch records: {len(launch_history)}")
launch_history.head(5)

Historical launch records: 101


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing _Outcome
0,04-06-2010,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,08-12-2010,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,22-05-2012,07:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,08-10-2012,00:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,01-03-2013,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [4]:
launch_history["Date"] = pd.to_datetime(launch_history["Date"], dayfirst=True)
launch_history["Landing _Outcome"] = (
    launch_history["Landing _Outcome"].astype(str).str.strip()
)

audit = pd.Series(
    {
        "rows": len(launch_history),
        "first_launch_year": int(launch_history["Date"].dt.year.min()),
        "latest_launch_year": int(launch_history["Date"].dt.year.max()),
        "launch_sites": launch_history["Launch_Site"].nunique(),
        "missing_values": int(launch_history.isna().sum().sum()),
    },
    name="value",
)
audit.to_frame()

,value
rows,101
first_launch_year,2010
latest_launch_year,2020
launch_sites,4
missing_values,0


In [5]:
site_counts = (
    launch_history.groupby("Launch_Site")
    .size()
    .sort_values(ascending=False)
    .rename("launches")
    .to_frame()
)
site_counts

,launches
Launch_Site,
CCAFS SLC-40,34
CCAFS LC-40,26
KSC LC-39A,25
VAFB SLC-4E,16


### Result

The web-scraped snapshot supplies **101 historical rows** and four source launch-site labels.
It complements the API dataset with customer, mission and landing-outcome fields used in the
SQL analysis.